In [ ]:
import pandas as pd

datos = {
'PRODUCTO': ['ARROZ', 'JABON', 'ARROZ_INTEGRAL', 'AZUCAR', 'ACEITE'],
'CANTIDAD': [10,7,8,2,10], 'PRECIO':[1500,2000,1500,3000,2500],
'FECHA': ['2023-07-25','2023-07-25','2023-07-26','2023-07-26','2023-07-27'],

}

df = pd.DataFrame(datos)
print(df)




         PRODUCTO  CANTIDAD  PRECIO       FECHA
0           ARROZ        10    1500  2023-07-25
1           JABON         7    2000  2023-07-25
2  ARROZ_INTEGRAL         8    1500  2023-07-26
3          AZUCAR         2    3000  2023-07-26
4          ACEITE        10    2500  2023-07-27


In [ ]:
productos_por_cantidad = df.groupby('PRODUCTO')['CANTIDAD'].sum().reset_index()
producto_mas_vendido = productos_por_cantidad.loc[productos_por_cantidad['CANTIDAD'].idxmax()]
display(producto_mas_vendido)

,0
PRODUCTO,ACEITE
CANTIDAD,10


In [ ]:
productos_ordenados = productos_por_cantidad.sort_values(by='CANTIDAD', ascending=False)
display(productos_ordenados)

,PRODUCTO,CANTIDAD
0,ACEITE,10
1,ARROZ,10
2,ARROZ_INTEGRAL,8
4,JABON,7
3,AZUCAR,2


In [ ]:
# Calculate total sale for each entry
df['TOTAL_VENTA'] = df['CANTIDAD'] * df['PRECIO']

# Group by 'FECHA' and sum 'TOTAL_VENTA' to get daily sales
ventas_por_fecha = df.groupby('FECHA')['TOTAL_VENTA'].sum().reset_index()

display(ventas_por_fecha)

,FECHA,TOTAL_VENTA
0,2023-07-25,29000
1,2023-07-26,18000
2,2023-07-27,25000


In [ ]:
import sqlite3

# Connect to (or create) a SQLite database
conn = sqlite3.connect('ventas.db')
cursor = conn.cursor()

# Create a table if it doesn't exist
# The schema is based on the DataFrame columns
cursor.execute('''
CREATE TABLE IF NOT EXISTS ventas_productos (
    PRODUCTO TEXT,
    CANTIDAD INTEGER,
    PRECIO REAL,
    FECHA TEXT,
    TOTAL_VENTA REAL
)
''')

# Insert DataFrame records into the table
df.to_sql('ventas_productos', conn, if_exists='replace', index=False)

print("Database 'ventas.db' created and data inserted into 'ventas_productos' table.")

# Verify by fetching some data
print("\nData from SQLite table:")
for row in cursor.execute('SELECT * FROM ventas_productos LIMIT 5'):
    print(row)

# Close the connection
conn.close()

Database 'ventas.db' created and data inserted into 'ventas_productos' table.

Data from SQLite table:
('ARROZ', 10, 1500, '2023-07-25', 15000)
('JABON', 7, 2000, '2023-07-25', 14000)
('ARROZ_INTEGRAL', 8, 1500, '2023-07-26', 12000)
('AZUCAR', 2, 3000, '2023-07-26', 6000)
('ACEITE', 10, 2500, '2023-07-27', 25000)


In [ ]:
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('ventas.db')
cursor = conn.cursor()

# Query to get table information (pragma_table_info returns column details)
cursor.execute("PRAGMA table_info(ventas_productos)")

# Fetch all results
columns_info = cursor.fetchall()

# Extract and print column names
print("Encabezados de la tabla 'ventas_productos':")
for col in columns_info:
    print(col[1]) # col[1] contains the column name

# Close the connection
conn.close()

Encabezados de la tabla 'ventas_productos':
PRODUCTO
CANTIDAD
PRECIO
FECHA
TOTAL_VENTA


In [ ]:
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('ventas.db')
cursor = conn.cursor()

# Select all records from the ventas_productos table
cursor.execute('SELECT * FROM ventas_productos')

# Fetch all results
all_records = cursor.fetchall()

# Print the records
print("Todos los registros de la tabla 'ventas_productos':")
for record in all_records:
    print(record)

# Close the connection
conn.close()

Todos los registros de la tabla 'ventas_productos':
('ARROZ', 10, 1500, '2023-07-25', 15000)
('JABON', 7, 2000, '2023-07-25', 14000)
('ARROZ_INTEGRAL', 8, 1500, '2023-07-26', 12000)
('AZUCAR', 2, 3000, '2023-07-26', 6000)
('ACEITE', 10, 2500, '2023-07-27', 25000)


In [ ]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect('ventas.db')

# Read the entire table into a pandas DataFrame
df_from_db = pd.read_sql_query("SELECT * FROM ventas_productos", conn)

# Close the connection
conn.close()

# Display the DataFrame for an aesthetic view
print("Datos de la tabla 'ventas_productos' en formato DataFrame:")
display(df_from_db)

Datos de la tabla 'ventas_productos' en formato DataFrame:


,PRODUCTO,CANTIDAD,PRECIO,FECHA,TOTAL_VENTA
0,ARROZ,10,1500,2023-07-25,15000
1,JABON,7,2000,2023-07-25,14000
2,ARROZ_INTEGRAL,8,1500,2023-07-26,12000
3,AZUCAR,2,3000,2023-07-26,6000
4,ACEITE,10,2500,2023-07-27,25000


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import sqlite3
import datetime

# Input widgets
producto_input = widgets.Text(description='Producto:')
cantidad_input = widgets.IntText(description='Cantidad:')
precio_input = widgets.FloatText(description='Precio:')
fecha_input = widgets.Text(description='Fecha (YYYY-MM-DD):', value=datetime.date.today().strftime('%Y-%m-%d'))

# Button to submit
insert_button = widgets.Button(description='Insertar Registro')

# Output widget to display messages
output_message = widgets.Output()

# Function to handle button click
def on_insert_button_clicked(b):
    with output_message:
        clear_output()
        try:
            producto = producto_input.value
            cantidad = cantidad_input.value
            precio = precio_input.value
            fecha = fecha_input.value

            # Basic validation
            if not producto or cantidad <= 0 or precio <= 0 or not fecha:
                print("Por favor, complete todos los campos y asegúrese de que cantidad y precio sean mayores que 0.")
                return

            # Calculate total sale
            total_venta = cantidad * precio

            # Connect to the database
            conn = sqlite3.connect('ventas.db')
            cursor = conn.cursor()

            # Insert data
            cursor.execute("INSERT INTO ventas_productos (PRODUCTO, CANTIDAD, PRECIO, FECHA, TOTAL_VENTA) VALUES (?, ?, ?, ?, ?)",
                           (producto, cantidad, precio, fecha, total_venta))
            conn.commit()
            conn.close()

            print(f"Registro insertado exitosamente: Producto='{producto}', Cantidad={cantidad}, Precio={precio}, Fecha='{fecha}', Total Venta={total_venta}")

            # Optionally, clear input fields after successful insertion
            # producto_input.value = ''
            # cantidad_input.value = 0
            # precio_input.value = 0.0
            # fecha_input.value = datetime.date.today().strftime('%Y-%m-%d') # Reset to current date

        except ValueError:
            print("Error de formato en la entrada. Asegúrese de que la fecha sea YYYY-MM-DD y los números sean válidos.")
        except Exception as e:
            print(f"Ocurrió un error: {e}")

# Link button click to function
insert_button.on_click(on_insert_button_clicked)

# Display the interface
print("Interfaz para insertar nuevos registros de ventas:")
display(widgets.VBox([
    producto_input,
    cantidad_input,
    precio_input,
    fecha_input,
    insert_button,
    output_message
]))

Interfaz para insertar nuevos registros de ventas:


Después de insertar un nuevo registro, puedes ejecutar la siguiente celda para ver la base de datos actualizada y confirmar que el nuevo registro ha sido añadido.

In [ ]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect('ventas.db')

# Read the entire table into a pandas DataFrame
df_updated = pd.read_sql_query("SELECT * FROM ventas_productos", conn)

# Close the connection
conn.close()

# Display the DataFrame
print("Datos actualizados de la tabla 'ventas_productos':")
display(df_updated)

Datos actualizados de la tabla 'ventas_productos':


,PRODUCTO,CANTIDAD,PRECIO,FECHA,TOTAL_VENTA
0,ARROZ,10,1500,2023-07-25,15000
1,JABON,7,2000,2023-07-25,14000
2,ARROZ_INTEGRAL,8,1500,2023-07-26,12000
3,AZUCAR,2,3000,2023-07-26,6000
4,ACEITE,10,2500,2023-07-27,25000


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dropdown para seleccionar la columna de ordenación
sort_column_selector = widgets.Dropdown(
    options=['PRODUCTO', 'CANTIDAD', 'PRECIO', 'FECHA'],
    value='PRODUCTO',
    description='Ordenar por:'
)

# Botón para aplicar la ordenación
sort_button = widgets.Button(description='Ordenar DataFrame')

# Output widget para mostrar el resultado
sort_output = widgets.Output()

# Función para manejar el clic del botón de ordenación
def on_sort_button_clicked(b):
    with sort_output:
        clear_output()
        column_to_sort = sort_column_selector.value

        # Convert 'FECHA' to datetime for proper sorting if it's not already
        if column_to_sort == 'FECHA':
            df_sorted = df_updated.sort_values(by=column_to_sort, ascending=True)
        elif column_to_sort in ['CANTIDAD', 'PRECIO']:
            # Sort numerical columns in descending order for common analysis
            df_sorted = df_updated.sort_values(by=column_to_sort, ascending=False)
        else:
            # Default to ascending for other columns (like PRODUCTO)
            df_sorted = df_updated.sort_values(by=column_to_sort, ascending=True)

        print(f"DataFrame ordenado por '{column_to_sort}':")
        display(df_sorted)

# Asocia la función al evento de clic del botón
sort_button.on_click(on_sort_button_clicked)

# Muestra los widgets
print("Menú para ordenar el DataFrame 'df_updated':")
display(widgets.VBox([
    sort_column_selector,
    sort_button,
    sort_output
]))

Menú para ordenar el DataFrame 'df_updated':
